In [41]:
from helper.ingest import load_faq_data, build_index
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI
from helper.evaluation_utils import llm_structured, calc_price,llm_structured_retry, map_progress,RAGWithUsage,calc_total_price
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json
from toyaikit.llm import OpenAIClient
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from sqlitesearch import TextSearchIndex
import numpy as np  
from helper.embedder import Embedder
from sqlitesearch import VectorSearchIndex
import time


In [34]:
load_dotenv()
openai_client = OpenAI()
embed = Embedder()

In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]


In [5]:
len(documents)

72

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [13]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [14]:


ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1130),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=115, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1401),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1853)]

In [16]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/72 [00:00<?, ?it/s]

In [17]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

360

In [18]:
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.11416500000000002

In [19]:
df_ground_truth = pd.DataFrame(ground_truth)

In [20]:
df_ground_truth.to_csv("data/ground_truth_assessment.csv", index=False)

In [21]:
df_ground_truth

,question,document
0,"What is the main goal of this module, and what...",01-agentic-rag/lessons/01-intro.md
1,How does this course treat large language mode...,01-agentic-rag/lessons/01-intro.md
2,What are the main limitations of LLMs that mak...,01-agentic-rag/lessons/01-intro.md
3,How does RAG help an LLM answer questions bett...,01-agentic-rag/lessons/01-intro.md
4,What will be covered in the first part of the ...,01-agentic-rag/lessons/01-intro.md
...,...,...
355,How do you handle retrieval when your source m...,07-project-example/lessons/07-chunking.md
356,"If I have one long transcript or PDF, what’s t...",07-project-example/lessons/07-chunking.md
357,"For books or other really long content, should...",07-project-example/lessons/07-chunking.md
358,How are images and slide decks supposed to be ...,07-project-example/lessons/07-chunking.md


In [81]:
df_ground = pd.read_csv("data/ground-truth.csv")
ground_truth_old = df_ground.to_dict(orient="records")

In [22]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [25]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [26]:
len(chunks)

295

## TEXT SEARCH

In [69]:
text_index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="homework_md.db"
)

for doc in chunks:
    text_index.add(doc)
    print(f"""Added: {doc["filename"]}...""")
    time.sleep(0.5)

text_index.close()
print("Done. Index saved to md.db")

Added: 01-agentic-rag/lessons/01-intro.md...
Added: 01-agentic-rag/lessons/01-intro.md...
Added: 01-agentic-rag/lessons/01-intro.md...
Added: 01-agentic-rag/lessons/02-environment.md...
Added: 01-agentic-rag/lessons/02-environment.md...
Added: 01-agentic-rag/lessons/02-environment.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/04-dataset.md...
Added: 01-agentic-rag/lessons/04-dataset.md...
Added: 01-agentic-rag/lessons/04-dataset.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/06-b

In [77]:
sqlite_text_index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="homework_md.db"
)

## VECTOR SEARCH

In [36]:
texts = []

for doc in chunks:
    text = doc["content"]
    texts.append(text)

In [38]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

295

In [42]:
X = np.array(vectors)
X.shape

(295, 384)

In [49]:
vector_index = VectorSearchIndex(
    keyword_fields=["filename"],
    db_path="homework_vector.db"
)

vector_index.fit(X, chunks)

vector_index.close()


In [50]:
sqlite_vector_index = VectorSearchIndex(
    keyword_fields=["filename"],
    db_path="homework_vector.db"
)

In [92]:
query = ground_truth_old[0]["question"]
query_vector = embed.encode(query)

results = sqlite_vector_index.search(query_vector, num_results=5)

In [93]:
results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

### HYBRID SEARCH

In [52]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [138]:
def hybrid_search(query, k=60):
    text_results = sqlite_text_index.search(query, num_results=10)
    vector_query = embed.encode(query)
    vector_results = sqlite_vector_index.search(vector_query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [139]:
q = ground_truth_old[0]["question"]

In [94]:
ground_truth_old[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [89]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [90]:
results = sqlite_text_index.search(q, num_results=5)

In [91]:
results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

### RELEVANCE

In [129]:
def text_search(query,k=None):
    return sqlite_text_index.search(
        query,
        num_results=5
    )


def vector_search(query,k=None):

    query_vector = embed.encode(query)
    
    return sqlite_vector_index.search(
        query_vector,
        num_results=5
    )

In [131]:
def compute_relevance(q, search_function,k=None):
    doc_id = q["filename"]
    results = search_function(query=q["question"],k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [132]:
def compute_relevance_total(ground_truth, search_function,k=None):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function,k)
        relevance_total.append(relevance)

    return relevance_total

In [117]:
relevance_total = compute_relevance_total(ground_truth_old, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [118]:
relevance_total 

[[0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 1, 0, 1, 0],
 [1, 0, 0, 1, 0],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 0, 0, 1],
 [1, 1, 1, 1, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [0, 0, 0, 1, 1],
 [0, 0, 1, 1, 0],
 [1, 1, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [1, 1, 0, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [119]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [120]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [134]:
def evaluate(ground_truth, search_function,k=None):
    relevance_total = compute_relevance_total(ground_truth, search_function,k)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [123]:
evaluate(
    ground_truth_old,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.775, 'mrr': 0.6297222222222221}

In [126]:
evaluate(
    ground_truth_old,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.5138888888888888, 'mrr': 0.40199074074074076}

In [141]:
for k in [1.0, 50.0, 100.0, 200.0]:
    result = evaluate(
        ground_truth_old,
        hybrid_search,
        k
    )
    print(f"k={k}: {result}")   

  0%|          | 0/360 [00:00<?, ?it/s]

k=1.0: {'hit_rate': 0.8027777777777778, 'mrr': 0.6241666666666669}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50.0: {'hit_rate': 0.8055555555555556, 'mrr': 0.5898148148148151}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100.0: {'hit_rate': 0.8055555555555556, 'mrr': 0.5898148148148151}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200.0: {'hit_rate': 0.8055555555555556, 'mrr': 0.5898148148148151}
